In [1]:
import sys
sys.path.append('../')
import os
os.environ['PATH'] = os.environ['PATH'] + os.pathsep + '/home/ubuntu/miniconda3/envs/utrtool/bin/'

import pybedtools

from utrcalling import utr_reference as utr_ref

In [2]:
gtf_path = "../tests/test_data/test_mus_musculus_GRCm38_102.gtf"
%time utr = utr_ref.extract_feature(gtf_path, 'three_prime_utr')
print(utr)

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'transcript_id', 'transcript_version', 'exon_number', 'gene_name', 'gene_source', 'gene_biotype', 'havana_gene', 'havana_gene_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'protein_id', 'protein_version', 'tag', 'transcript_support_level', 'ccds_id', 'exon_id', 'exon_version', 'havana_transcript', 'havana_transcript_version']
CPU times: user 98 ms, sys: 4.03 ms, total: 102 ms
Wall time: 100 ms
     seqname          source          feature      start        end  score  \
4          1         ensembl  three_prime_utr  130697017  130697884    NaN   
5          1         ensembl  three_prime_utr  170885460  170885881    NaN   
6          1         ensembl  three_prime_utr   77367185   77369671    NaN   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    NaN   
22         1  ensembl_havana  three_prime_utr  128412118  128412347    NaN   
...      ...             ...              ...

In [3]:
feature_list = utr["feature"].unique()
print(feature_list)

['three_prime_utr']


In [4]:
utr['seqname'].unique()

array(['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
       '2', '3', '4', '5', '6', '7', '8', '9', 'X', 'Y'], dtype=object)

In [5]:
utr = utr.sort_values(by=['seqname', 'start'])  # sort by chr and start for bed merge
utr.fillna(value={'score': 0}, inplace=True)
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [6]:
# ann_remove = ['seqname', 'start', 'end']
annotations = ['gene_id', 'score', 'strand', 'transcript_id']
columns, ann_idx = utr_ref.prepare_annotations_index(annotations)
utr_bed = pybedtools.BedTool.from_dataframe(utr, columns=columns)
utr_bed.saveas('testbed.bed')
utr_reduced = utr_ref.reduce_regions(utr_bed,
annotations_index=ann_idx)  # find and merge overlapping regions


In [7]:
print(utr)

     seqname          source          feature      start        end  score  \
100        1          havana  three_prime_utr    7160892    7161063    0.0   
107        1          havana  three_prime_utr    9552943    9552976    0.0   
108        1          havana  three_prime_utr    9553318    9553472    0.0   
21         1  ensembl_havana  three_prime_utr   10024601   10024978    0.0   
70         1          havana  three_prime_utr   15843083   15843470    0.0   
...      ...             ...              ...        ...        ...    ...   
1980       X          havana  three_prime_utr  166189501  166189820    0.0   
1981       X          havana  three_prime_utr  167476787  167478076    0.0   
1982       X          havana  three_prime_utr  167497813  167497949    0.0   
1950       X  ensembl_havana  three_prime_utr  169776378  169776476    0.0   
1999       Y          havana  three_prime_utr    1224497    1224546    0.0   

     strand frame             gene_id gene_version  ...  \
100 

In [8]:
print(utr_reduced.head())

1	7160892	7161063	ENSMUSG00000051285	0.0	+	ENSMUST00000182675	1
 1	9552943	9552976	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	1
 1	9553318	9553472	ENSMUSG00000025911	0.0	+	ENSMUST00000130927	1
 1	10024601	10024978	ENSMUSG00000025917	0.0	-	ENSMUST00000027050	1
 1	15843083	15843470	ENSMUSG00000025925	0.0	+	ENSMUST00000027057	1
 1	30944938	30945030	ENSMUSG00000117310	0.0	-	ENSMUST00000233506	1
 1	33734702	33734858	ENSMUSG00000004768	0.0	+	ENSMUST00000138024	1
 1	34552334	34552678	ENSMUSG00000037529	0.0	-	ENSMUST00000190790	1
 1	34811830	34812739	ENSMUSG00000037509	0.0	+	ENSMUST00000159747	1
 1	36428102	36428216	ENSMUSG00000001143	0.0	-	ENSMUST00000001171	1
 None


In [9]:
utr_reduced.saveas('testbed_merged.bed')

<BedTool(testbed_merged.bed)>

In [10]:
from utrcalling import utr_mapping

# check if aligments are correctly parsed: (each should output True)
print(all(alignment.proper_pair for alignment in alignments))
print(not any(alignment.pcr_or_optical_duplicate for alignment in alignments))
print(not any(alignment.not_primary_alignment for alignment in alignments))

In [11]:
intervals = utr_mapping.generate_utr_intervals(utr_reduced)
ct = 0
for name, interval in intervals.items(): 
    print(name, interval, interval.start_d, sep=': ')
    ct += 1
    if ct == 5:
        break

ENSMUSG00000051285: 1:[7160891,7161063)/+: 7160891
ENSMUSG00000025911: 1:[9552942,9552976)/+: 9552942
ENSMUSG00000025911_1: 1:[9553317,9553472)/+: 9553317
ENSMUSG00000025917: 1:[10024600,10024978)/-: 10024977
ENSMUSG00000025925: 1:[15843082,15843470)/+: 15843082


In [12]:
print(intervals['ENSMUSG00000005804'], intervals['ENSMUSG00000025917'], sep='\n')

2:[122746630,122749475)/+
1:[10024600,10024978)/-


In [13]:
alignments = ["../tests/test_data/test_UTRoldNSCgenome_bwt2_102_sort.bam", "../tests/test_data/test_UTRyoungNSCgenome_bwt2_102_sort.bam"]
read_mappings = [utr_mapping.map_reads_to_utr(alignment, intervals, read='first', version=2) for alignment in alignments]

In [14]:
print(read_mappings[0].ENSMUSG00000005804)

MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748098, 122748088, 122748096], read_strands=['-', '-', '-'], read_sequences=['TCACCCATCAGGCATTTTATTGTAACATTTTGCAAGGTAATGACATTGTCC', 'GGCATTTTATTGTAACATTTTGCAAGGTAATGACATT', 'AGCTATCAGGCATTTTATTGTAACATTTTGCAAGGTAATGACAT'], read_barcodes=['GATGAAACATTAACCG', 'TGCGCAGAGTCCATAC', 'GCTGGGTGTACGCACC'], read_umis=['CAACACAAG', 'ATCGAGAGT', 'CTGAGTATG'], read_lengths=[51, 37, 44], utr_sizes_from_reads=[1468, 1458, 1466])


In [15]:
dct = read_mappings[0].todict()
for key, value in dct.items():
    print(key, value, sep='\n')
    break


ENSMUSG00000025917
{'utr_name': 'ENSMUSG00000025917', 'read_start_positions': [10024824, 10024829, 10024829, 10024833], 'read_strands': ['+', '+', '+', '+'], 'read_sequences': ['GACAATTTCACCTGAGAGTTTTACTACATCGCTCATACGCAGGACA', 'TTTGACGGGATAGTTTTATTACAGTGCTAATACTCAGGACCCCCCCCAGC', 'TTTCACGGGATAGTTTTATTTCAGTGCTAATACCCAGGACACCACACA', 'ACGGGCGAGTTTTATTACAGTGCAAATACTCAGGACACCACACAGC'], 'read_barcodes': ['CGTTGGGCAGTCAGAG', 'CCGTGGAGTCTCGTTC', 'GTAGTCAAGCTATGCT', 'TGCGCAGAGGCATGTG'], 'read_umis': ['TGGCTTATA', 'AACGTAACA', 'ACCTCCGGT', 'ACCCGCACT'], 'read_lengths': [46, 50, 48, 46], 'utr_sizes_from_reads': [153, 148, 148, 144]}


In [16]:
read_mappings[1]['ENSMUSG00000005804', 'ENSMUSG00000025917']

[MappedReadInformation(utr_name='ENSMUSG00000005804', read_start_positions=[122748095, 122748095], read_strands=['-', '-'], read_sequences=['GCTATACGGCATTTTATTGAAACATTTTGCGAGGTAATGCCATT', 'GCTATTAGGCCTTTTATTGTAACATTTTGCAGGGTAATGACAT'], read_barcodes=['AGACGTTAGTTGCAGG', 'GGCGACTGTAGCAAAT'], read_umis=['CTCATGGTA', 'ACCGGGTCC'], read_lengths=[44, 43], utr_sizes_from_reads=[1465, 1465]),
 MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+'], read_sequences=['AAGTCAGATTATGTTTTATTTGTAGTTGTATTACATTTTTTCT', 'CAAATTCACGGGATAGTTTTATAACAGTGCTCATACTCAGGACCC', 'TTTAACGGGATAGTTTTATTACAGTGCTAATACTCGGGACACCACACAGCACCT', 'TTTCGCGGGATAGTATTATTACAGTGCTAATACCCAGGACACCCCACAGC', 'GTGATAATTTTTTTACAGTGCTAATACACAGGACCCCCCAC', 'TAGTTTTTTTACAGTTCTAATACTCAGGACACCACACCGCA'], read_barcodes=['CGACCTTTCTGCTGCT', 'TGTTCCGAGGGTCGAT', 'GTACGTAAGCGTTTAC', 'GTGCGGTAGGACAGCT', 'GCTGCAGCACTGTGTA',

In [17]:
read_mappings[0][0:4:2]

['ENSMUSG00000025917', 'ENSMUSG00000037509']

In [18]:
from utrcalling import peak_calling

In [19]:
positions_counts_merged = peak_calling.merge_position_counts(read_mappings)
print(len(positions_counts_merged), positions_counts_merged.ENSMUSG00000025917, sep='\n\n')

71

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_sequences=['GACAATTTCACCTGAGAGTTTTACTACATCGCTCATACGCAGGACA', 'TTTGACGGGATAGTTTTATTACAGTGCTAATACTCAGGACCCCCCCCAGC', 'TTTCACGGGATAGTTTTATTTCAGTGCTAATACCCAGGACACCACACA', 'ACGGGCGAGTTTTATTACAGTGCAAATACTCAGGACACCACACAGC', 'AAGTCAGATTATGTTTTATTTGTAGTTGTATTACATTTTTTCT', 'CAAATTCACGGGATAGTTTTATAACAGTGCTCATACTCAGGACCC', 'TTTAACGGGATAGTTTTATTACAGTGCTAATACTCGGGACACCACACAGCACCT', 'TTTCGCGGGATAGTATTATTACAGTGCTAATACCCAGGACACCCCACAGC', 'GTGATAATTTTTTTACAGTGCTAATACACAGGACCCCCCAC', 'TAGTTTTTTTACAGTTCTAATACTCAGGACACCACACCGCA'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']

In [20]:
utr_subsetted = peak_calling.subset_data(positions_counts_merged, utr_reduced)
print(f'sizes: subset: {len(utr_subsetted)}, original: {len(utr_reduced)}')


sizes: subset: 71, original: 1016


In [21]:
utr_subsetted[0].fields

['1',
 '10024601',
 '10024978',
 'ENSMUSG00000025917',
 '0.0',
 '-',
 'ENSMUST00000027050',
 '1']

In [22]:
%time ft = peak_calling.filter_positions(positions_counts_merged)

CPU times: user 16.1 ms, sys: 0 ns, total: 16.1 ms
Wall time: 15.7 ms


In [23]:
import pandas as pd
vivo_anno_path = "../tests/test_data/vivo_anno.csv"
vivo_anno = pd.read_csv(vivo_anno_path)

In [24]:
vivo_anno.head()

,CellBarcode,CellType,CellAge,pTime
0,AAACCTGCACCAGTTA-1,qNSC1,old,4.018431
1,AAACCTGCATGGTCAT-1,qNSC2,old,14.861303
2,AAACGGGCACCGGAAA-1,qNSC2,old,20.304435
3,AAACGGGCAGCAGTTT-1,qNSC2,old,10.686136
4,AAACGGGGTAAGTTCC-1,qNSC2,old,10.351203


In [25]:
len(ft)

71

In [26]:
ft.ENSMUSG00000025917

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_sequences=['GACAATTTCACCTGAGAGTTTTACTACATCGCTCATACGCAGGACA', 'TTTGACGGGATAGTTTTATTACAGTGCTAATACTCAGGACCCCCCCCAGC', 'TTTCACGGGATAGTTTTATTTCAGTGCTAATACCCAGGACACCACACA', 'ACGGGCGAGTTTTATTACAGTGCAAATACTCAGGACACCACACAGC', 'AAGTCAGATTATGTTTTATTTGTAGTTGTATTACATTTTTTCT', 'CAAATTCACGGGATAGTTTTATAACAGTGCTCATACTCAGGACCC', 'TTTAACGGGATAGTTTTATTACAGTGCTAATACTCGGGACACCACACAGCACCT', 'TTTCGCGGGATAGTATTATTACAGTGCTAATACCCAGGACACCCCACAGC', 'GTGATAATTTTTTTACAGTGCTAATACACAGGACCCCCCAC', 'TAGTTTTTTTACAGTTCTAATACTCAGGACACCACACCGCA'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], re

In [27]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in ft.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

32

71


In [28]:
filts = peak_calling.call_peaks(ft, vivo_anno_path)
len(filts)

71

In [29]:
counter = 0
control = 0
barcodes_real = vivo_anno.CellBarcode
for barcodes in filts.read_barcodes:
    control +=1
    if set(barcodes).issubset(barcodes_real):
        counter += 1
print(counter, control, sep='\n\n')

71

71


In [30]:
print(positions_counts_merged.ENSMUSG00000025917, ft.ENSMUSG00000025917, filts.ENSMUSG00000025917, sep='\n\n')

MappedReadInformation(utr_name='ENSMUSG00000025917', read_start_positions=[10024824, 10024829, 10024829, 10024833, 10024602, 10024826, 10024829, 10024829, 10024835, 10024839], read_strands=['+', '+', '+', '+', '+', '+', '+', '+', '+', '+'], read_sequences=['GACAATTTCACCTGAGAGTTTTACTACATCGCTCATACGCAGGACA', 'TTTGACGGGATAGTTTTATTACAGTGCTAATACTCAGGACCCCCCCCAGC', 'TTTCACGGGATAGTTTTATTTCAGTGCTAATACCCAGGACACCACACA', 'ACGGGCGAGTTTTATTACAGTGCAAATACTCAGGACACCACACAGC', 'AAGTCAGATTATGTTTTATTTGTAGTTGTATTACATTTTTTCT', 'CAAATTCACGGGATAGTTTTATAACAGTGCTCATACTCAGGACCC', 'TTTAACGGGATAGTTTTATTACAGTGCTAATACTCGGGACACCACACAGCACCT', 'TTTCGCGGGATAGTATTATTACAGTGCTAATACCCAGGACACCCCACAGC', 'GTGATAATTTTTTTACAGTGCTAATACACAGGACCCCCCAC', 'TAGTTTTTTTACAGTTCTAATACTCAGGACACCACACCGCA'], read_barcodes=['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2'], re

In [31]:
print(ft.ENSMUSG00000025917['read_barcodes'], filts.ENSMUSG00000025917['read_barcodes'], sep='\n\n')

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'GTAGTCAAGCTATGCT-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']

['CGTTGGGCAGTCAGAG-1', 'CCGTGGAGTCTCGTTC-1', 'TGCGCAGAGGCATGTG-1', 'CGACCTTTCTGCTGCT-2', 'TGTTCCGAGGGTCGAT-2', 'GTACGTAAGCGTTTAC-2', 'GTGCGGTAGGACAGCT-2', 'GCTGCAGCACTGTGTA-2', 'GTCCTCAGTAAACGCG-2']
